In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import ListedColormap


from util import *

sns.set_palette('colorblind')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
filepath = '/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/saturation'

In [3]:
unique_timestamps = []

for filename in os.listdir(filepath):
    timestamp = filename.split('_data_')[1].split('.json')[0]
    unique_timestamps.append(timestamp)

unique_timestamps = list(set(unique_timestamps))
unique_timestamps

['2025-12-24 04:33:00',
 '2025-12-23 15:56:44',
 '2025-12-23 12:27:23',
 '2025-12-22_17:45:33',
 '2025-12-16 11:30:02',
 '2025-12-23 21:33:56',
 '2025-12-23 12:27:14',
 '2025-12-31 14:32:30',
 '2025-12-22_20:38:29',
 '2026-01-01 00:58:55',
 '2025-12-31 17:37:07',
 '2026-01-01 10:36:33',
 '2025-12-24 08:33:00',
 '2025-12-23 09:52:08',
 '2025-12-31 18:24:09',
 '2025-12-24 00:33:00',
 '2025-12-22_20:38:54',
 '2025-12-31 21:31:40',
 '2025-12-23 15:56:36',
 '2025-12-23 09:52:32']

In [4]:
# remove files that dont have both metrics and timestamps

num_files = {}
for filename in os.listdir(filepath):
    timestamp = filename.split('_data_')[1].split('.json')[0]

    if timestamp in unique_timestamps:
        if timestamp not in num_files:
            num_files[timestamp] = 0

        num_files[timestamp] += 1
        
remove_list = set()

for timestamp, count in num_files.items():
    if count != 2:
        remove_list.add(timestamp)

for x in remove_list:
    unique_timestamps.remove(x)
unique_timestamps


['2025-12-24 04:33:00',
 '2025-12-16 11:30:02',
 '2025-12-23 21:33:56',
 '2025-12-31 14:32:30',
 '2026-01-01 00:58:55',
 '2025-12-31 17:37:07',
 '2026-01-01 10:36:33',
 '2025-12-24 08:33:00',
 '2025-12-31 18:24:09',
 '2025-12-24 00:33:00',
 '2025-12-31 21:31:40']

In [5]:
required_files = {}
for filename in os.listdir(filepath):
    timestamp = filename.split('_data_')[1].split('.json')[0]

    if timestamp in unique_timestamps:
        if timestamp not in required_files:
            required_files[timestamp] = []

        required_files[timestamp].append(filename)

required_files

{'2026-01-01 00:58:55': ['control_metrics_data_2026-01-01 00:58:55',
  'control_timestamps_data_2026-01-01 00:58:55'],
 '2025-12-24 00:33:00': ['control_metrics_data_2025-12-24 00:33:00',
  'control_timestamps_data_2025-12-24 00:33:00'],
 '2025-12-31 14:32:30': ['control_metrics_data_2025-12-31 14:32:30',
  'control_timestamps_data_2025-12-31 14:32:30'],
 '2025-12-16 11:30:02': ['metrics_data_2025-12-16 11:30:02.json',
  'timestamps_data_2025-12-16 11:30:02.json'],
 '2025-12-24 04:33:00': ['control_metrics_data_2025-12-24 04:33:00',
  'control_timestamps_data_2025-12-24 04:33:00'],
 '2025-12-24 08:33:00': ['control_timestamps_data_2025-12-24 08:33:00',
  'control_metrics_data_2025-12-24 08:33:00'],
 '2025-12-31 18:24:09': ['control_metrics_data_2025-12-31 18:24:09',
  'control_timestamps_data_2025-12-31 18:24:09'],
 '2025-12-31 21:31:40': ['control_metrics_data_2025-12-31 21:31:40',
  'control_timestamps_data_2025-12-31 21:31:40'],
 '2025-12-31 17:37:07': ['control_metrics_data_2025-12

In [6]:
merged_df = pd.DataFrame()

for timestamp, files in required_files.items():
    for filename in files:
        if 'metrics' in filename:
            with open(f'{filepath}/{filename}', 'r') as fp:
                metrics_data = json.load(fp)
        elif 'timestamps' in filename:
            with open(f'{filepath}/{filename}', 'r') as fp:
                timestamps_data = json.load(fp)
        else:
            raise('File not matching req')
        
    metrics_df = (
        pd.DataFrame.from_dict(metrics_data, orient="index", columns=[f"likes_{timestamp}", f"comments_{timestamp}"])
        .reset_index()
        .rename(columns={"index": "urlid"})
    )

    timestamps_df = (
        pd.DataFrame.from_dict(timestamps_data, orient="index", columns=[f"timestamps_{timestamp}"])
        .reset_index()
        .rename(columns={"index": "urlid"})
    )

    temp_df = pd.merge(metrics_df, timestamps_df, on='urlid', how='outer')

    if merged_df.empty:
        merged_df = temp_df.copy()
    else:
        merged_df = pd.merge(merged_df, temp_df,  on='urlid', how='outer')

In [7]:
display(merged_df.shape)
merged_df['urlid'].duplicated().sum()

(669, 34)

np.int64(0)

In [8]:
merged_df.drop(columns=[x for x in merged_df.columns if 'likes' in x], inplace=True)
merged_df

,urlid,comments_2026-01-01 00:58:55,timestamps_2026-01-01 00:58:55,comments_2025-12-24 00:33:00,timestamps_2025-12-24 00:33:00,comments_2025-12-31 14:32:30,timestamps_2025-12-31 14:32:30,comments_2025-12-16 11:30:02,timestamps_2025-12-16 11:30:02,comments_2025-12-24 04:33:00,timestamps_2025-12-24 04:33:00,comments_2025-12-24 08:33:00,timestamps_2025-12-24 08:33:00,comments_2025-12-31 18:24:09,timestamps_2025-12-31 18:24:09,comments_2025-12-31 21:31:40,timestamps_2025-12-31 21:31:40,comments_2025-12-31 17:37:07,timestamps_2025-12-31 17:37:07,comments_2025-12-23 21:33:56,timestamps_2025-12-23 21:33:56,comments_2026-01-01 10:36:33,timestamps_2026-01-01 10:36:33
0,C_9aw1hPL87,1.4K,2024-09-16T01:32:29.000Z,NaN,NaN,NaN,NaN,1.4K,2024-09-16T01:32:29.000Z,NaN,NaN,NaN,NaN,NaN,2024-09-16T01:32:29.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DCLQ9rgJBZt,678,2024-11-10T03:41:05.000Z,NaN,NaN,NaN,NaN,665,2024-11-10T03:41:05.000Z,NaN,NaN,NaN,NaN,NaN,2024-11-10T03:41:05.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DEpc4-eN3Wz,1.5K,2025-01-10T14:01:37.000Z,1.4K,2025-01-10T14:01:37.000Z,1.5K,2025-01-10T14:01:37.000Z,1.4K,2025-01-10T14:01:37.000Z,1.4K,2025-01-10T14:01:37.000Z,NaN,NaN,NaN,2025-01-10T14:01:37.000Z,NaN,NaN,1.5K,2025-01-10T14:01:37.000Z,1.4K,2025-01-10T14:01:37.000Z,1.5K,2025-01-10T14:01:37.000Z
3,DGt0ml1tMXj,3.5K,2025-03-02T23:48:52.000Z,NaN,NaN,NaN,NaN,3.5K,2025-03-02T23:48:52.000Z,NaN,NaN,NaN,NaN,NaN,2025-03-02T23:48:52.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DMH6nwUShIT,38,2025-07-15T09:38:21.000Z,NaN,NaN,38,2025-07-15T09:38:21.000Z,38,2025-07-15T09:38:21.000Z,NaN,NaN,NaN,NaN,NaN,2025-07-15T09:38:21.000Z,NaN,2025-07-15T09:38:21.000Z,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,DSxdf_XlE6N,None,2025-12-27T16:00:34.000Z,NaN,NaN,None,2025-12-27T16:00:34.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-27T16:00:34.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
665,DSz1HZRkgj4,53,2025-12-28T14:05:14.000Z,NaN,NaN,53,2025-12-28T14:05:14.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-28T14:05:14.000Z,NaN,2025-12-28T14:05:14.000Z,NaN,NaN,NaN,NaN,53,2025-12-28T14:05:14.000Z
666,DSzCPvUETwL,NaN,NaN,NaN,NaN,105,2025-12-28T06:40:41.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-28T06:40:41.000Z,NaN,2025-12-28T06:40:41.000Z,NaN,NaN,NaN,NaN,NaN,NaN
667,DSza9rakoIg,209,2025-12-28T10:16:40.000Z,NaN,NaN,204,2025-12-28T10:16:40.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-28T10:16:40.000Z,NaN,2025-12-28T10:16:40.000Z,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
timestamps_cols = [x for x in merged_df.columns if 'timestamps' in x]
merged_df[timestamps_cols] = merged_df[timestamps_cols].apply(pd.to_datetime, errors="coerce")

merged_df["timestamp"] = merged_df[timestamps_cols].bfill(axis=1).iloc[:, 0]
non_null_cnt = merged_df[timestamps_cols].notna().sum(axis=1)
n_unique_non_null = merged_df[timestamps_cols].apply(lambda r: r.dropna().nunique(), axis=1)
merged_df["timestamp_conflict"] = (non_null_cnt > 1) & (n_unique_non_null > 1)
conflicts = merged_df.loc[merged_df["timestamp_conflict"], timestamps_cols + ["timestamp"]]
conflicts

/var/folders/sh/fhrqk28n1bzd27x7z4p52c940000gn/T/ipykernel_83109/1772047738.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df["timestamp"] = merged_df[timestamps_cols].bfill(axis=1).iloc[:, 0]


,timestamps_2026-01-01 00:58:55,timestamps_2025-12-24 00:33:00,timestamps_2025-12-31 14:32:30,timestamps_2025-12-16 11:30:02,timestamps_2025-12-24 04:33:00,timestamps_2025-12-24 08:33:00,timestamps_2025-12-31 18:24:09,timestamps_2025-12-31 21:31:40,timestamps_2025-12-31 17:37:07,timestamps_2025-12-23 21:33:56,timestamps_2026-01-01 10:36:33,timestamp


In [10]:
merged_df.drop(columns=timestamps_cols, inplace=True)
merged_df.drop(columns=['timestamp_conflict'], inplace=True)

In [11]:
merged_df

,urlid,comments_2026-01-01 00:58:55,comments_2025-12-24 00:33:00,comments_2025-12-31 14:32:30,comments_2025-12-16 11:30:02,comments_2025-12-24 04:33:00,comments_2025-12-24 08:33:00,comments_2025-12-31 18:24:09,comments_2025-12-31 21:31:40,comments_2025-12-31 17:37:07,comments_2025-12-23 21:33:56,comments_2026-01-01 10:36:33,timestamp
0,C_9aw1hPL87,1.4K,NaN,NaN,1.4K,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-16 01:32:29+00:00
1,DCLQ9rgJBZt,678,NaN,NaN,665,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-10 03:41:05+00:00
2,DEpc4-eN3Wz,1.5K,1.4K,1.5K,1.4K,1.4K,NaN,NaN,NaN,1.5K,1.4K,1.5K,2025-01-10 14:01:37+00:00
3,DGt0ml1tMXj,3.5K,NaN,NaN,3.5K,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-03-02 23:48:52+00:00
4,DMH6nwUShIT,38,NaN,38,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-07-15 09:38:21+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,DSxdf_XlE6N,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-27 16:00:34+00:00
665,DSz1HZRkgj4,53,NaN,53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53,2025-12-28 14:05:14+00:00
666,DSzCPvUETwL,NaN,NaN,105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-28 06:40:41+00:00
667,DSza9rakoIg,209,NaN,204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-28 10:16:40+00:00


In [12]:
timestamps_cols = [x for x in merged_df.columns if x not in ['urlid', 'timestamp']]
timestamps_cols = sorted(timestamps_cols)
merged_df = merged_df[['urlid', 'timestamp'] + timestamps_cols]

In [13]:
drop_columns = ['comments_2025-12-16 11:30:02', 'comments_2025-12-23 21:33:56', 'comments_2025-12-24 00:33:00', 'comments_2025-12-24 04:33:00', 'comments_2025-12-24 08:33:00']
merged_df = merged_df.drop(columns=drop_columns)

In [14]:
merged_df.shape

(669, 8)

In [15]:
comments_cols = [x for x in merged_df.columns if x not in ['urlid', 'timestamp']]
merged_df = merged_df.dropna(subset=comments_cols, how='all')

In [16]:
merged_df.describe()

,urlid,timestamp,comments_2025-12-31 14:32:30,comments_2025-12-31 17:37:07,comments_2025-12-31 18:24:09,comments_2025-12-31 21:31:40,comments_2026-01-01 00:58:55,comments_2026-01-01 10:36:33
count,313,313,188,48,0,0,216,73
unique,313,NaN,136,45,0,0,143,65
top,C_9aw1hPL87,NaN,5,1.6K,NaN,NaN,1.4K,5
freq,1,NaN,7,2,NaN,NaN,5,4
mean,NaN,2025-12-09 04:02:10.226836992+00:00,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,2014-03-25 16:24:48+00:00,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,2025-12-29 23:03:38+00:00,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,2025-12-31 01:30:07+00:00,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,2025-12-31 18:01:49+00:00,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,2026-01-01 07:40:56+00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
merged_df = merged_df.drop(columns=['comments_2025-12-31 18:24:09', 'comments_2025-12-31 21:31:40'])

In [18]:
comments_cols = [x for x in merged_df.columns if x not in ['urlid', 'timestamp']]
for col in comments_cols:
    merged_df[col] = merged_df[col].apply(numify_metrics)

In [19]:
comments_cols = [x for x in merged_df.columns if x not in ['urlid', 'timestamp']]
timestamps = [x.split('comments_')[1] for x in comments_cols]

for timestamp in timestamps:
    mtimestamp = pd.to_datetime(timestamp).tz_localize("Europe/Berlin").tz_convert("UTC")
    print(mtimestamp)
    merged_df[f'time_diff_{timestamp}'] = (mtimestamp - merged_df['timestamp']).dt.total_seconds() / 3600
    merged_df[f'perc_growth_{timestamp}'] = merged_df.apply(lambda x: (x[f'comments_{timestamp}']*100)/x[f'time_diff_{timestamp}'] if pd.notna(x[f'comments_{timestamp}']) else None, axis=1)



2025-12-31 13:32:30+00:00
2025-12-31 16:37:07+00:00
2025-12-31 23:58:55+00:00
2026-01-01 09:36:33+00:00


In [20]:
threshold_date = pd.to_datetime('2025-12-31 0:0:0').tz_localize('utc')
merged_df = merged_df[merged_df['timestamp'] > threshold_date]

In [23]:
merged_df = merged_df.sort_values('timestamp')

In [26]:
merged_df.head()

,urlid,timestamp,comments_2025-12-31 14:32:30,comments_2025-12-31 17:37:07,comments_2026-01-01 00:58:55,comments_2026-01-01 10:36:33,time_diff_2025-12-31 14:32:30,perc_growth_2025-12-31 14:32:30,time_diff_2025-12-31 17:37:07,perc_growth_2025-12-31 17:37:07,time_diff_2026-01-01 00:58:55,perc_growth_2026-01-01 00:58:55,time_diff_2026-01-01 10:36:33,perc_growth_2026-01-01 10:36:33
62,DS4MuQriHYb,2025-12-31 00:00:02+00:00,3.0,NaN,3.0,NaN,13.541111,22.154755,16.618056,NaN,23.981389,12.509701,33.608611,NaN
68,DS51H70AQwk,2025-12-31 00:00:53+00:00,66.0,NaN,NaN,NaN,13.526944,487.915067,16.603889,NaN,23.967222,NaN,33.594444,NaN
69,DS51NxagfX6,2025-12-31 00:01:11+00:00,16.0,NaN,NaN,NaN,13.521944,118.326178,16.598889,NaN,23.962222,NaN,33.589444,NaN
130,DS6C6OxD109,2025-12-31 00:03:09+00:00,NaN,NaN,5.0,NaN,13.489167,NaN,16.566111,NaN,23.929444,20.894760,33.556667,NaN
73,DS54jzfgShK,2025-12-31 00:30:54+00:00,66.0,NaN,NaN,NaN,13.026667,506.653019,16.103611,NaN,23.466944,NaN,33.094167,NaN


In [27]:
merged_df = merged_df[['urlid', 'timestamp', 'comments_2025-12-31 14:32:30', 'time_diff_2025-12-31 14:32:30', 'perc_growth_2025-12-31 14:32:30',
                       'comments_2025-12-31 17:37:07', 'time_diff_2025-12-31 17:37:07', 'perc_growth_2025-12-31 17:37:07',
                       'comments_2026-01-01 00:58:55', 'time_diff_2026-01-01 00:58:55', 'perc_growth_2026-01-01 00:58:55',
                       'comments_2026-01-01 10:36:33', 'time_diff_2026-01-01 10:36:33', 'perc_growth_2026-01-01 10:36:33']]

In [29]:
merged_df.to_clipboard()

### analysing merged data 

In [2]:
df = pd.read_excel('/Users/brahmaninutakki/saarland/insta-comments/saved_data/News Sites US.xlsx', sheet_name='Sheet17')

snapshots = [
    ("comments_2025-12-31 14:32:30", "time_diff_2025-12-31 14:32:30"),
    ("comments_2025-12-31 17:37:07", "time_diff_2025-12-31 17:37:07"),
    ("comments_2026-01-01 00:58:55", "time_diff_2026-01-01 00:58:55"),
    ("comments_2026-01-01 10:36:33", "time_diff_2026-01-01 10:36:33"),
]

for c, _ in snapshots:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce")
for _, t in snapshots:
    df[t] = pd.to_numeric(df[t], errors="coerce")

def sat_time(ts, vs, frac):
    ts = np.array(ts, float)
    vs = np.array(vs, float)
    vs = np.maximum.accumulate(vs)
    final = vs[-1]
    if len(ts) < 2 or not np.isfinite(final) or final <= 0:
        return np.nan
    target = frac * final
    if vs[0] >= target:
        return ts[0]
    idx = np.argmax(vs >= target)
    if vs[idx] < target:
        return np.nan
    t0, t1 = ts[idx-1], ts[idx]
    v0, v1 = vs[idx-1], vs[idx]
    if t1 == t0 or v1 == v0:
        return t1
    return t0 + (target - v0) * (t1 - t0) / (v1 - v0)

rows = []
for _, r in df.iterrows():
    pts = [(r[t], r[c]) for c, t in snapshots if pd.notna(r[c]) and pd.notna(r[t]) and r[t] >= 0]
    if len(pts) < 2:
        continue
    pts.sort()
    ts = [p[0] for p in pts]
    vs = [p[1] for p in pts]
    rows.append({
        "urlid": r["urlid"],
        "t_final": ts[-1],
        "comments_final": np.maximum.accumulate(vs)[-1],
        "t90": sat_time(ts, vs, 0.90),
        "t95": sat_time(ts, vs, 0.95),
        "t99": sat_time(ts, vs, 0.99),
    })

out = pd.DataFrame(rows)

out_20h = out[out["t_final"] >= 20]
print(out_20h[["t90","t95","t99"]].describe())


             t90        t95        t99
count  15.000000  15.000000  15.000000
mean   14.796911  17.688822  21.431866
std     4.538116   5.396810   7.391932
min     8.461273   9.592303  10.497127
25%    10.887181  13.854764  16.357842
50%    14.471957  18.505701  21.732696
75%    18.659244  21.366856  28.750342
max    22.743194  26.353403  29.882873


In [6]:
accounts_data = pd.read_excel('/Users/brahmaninutakki/saarland/insta-comments/saved_data/News Sites US.xlsx', sheet_name='Sheet21')
out = pd.merge(out, accounts_data, on='urlid', how='inner')
out.shape

,urlid,account
0,DS4MuQriHYb,peacock
1,DS6GU8bjwGW,washingtonpost
2,DS6IvCLExii,washtimes
3,DS6MrqFEsKc,catloversclub
4,DS6QWPnEQfH,msnbc


In [15]:
nonnews_accs = ['catloversclub', 'peacock', 'espn', 'thehill']
out['type'] = out['account'].apply(lambda x: 'Non-News' if x in nonnews_accs else 'News')

In [16]:
out.head()

,urlid,t_final,comments_final,t90,t95,t99,account,type
0,DS4MuQriHYb,23.981389,3.0,13.541111,13.541111,13.541111,peacock,Non-News
1,DS6GU8bjwGW,33.090556,336.0,15.369226,19.487133,27.699311,washingtonpost,News
2,DS6IvCLExii,23.064167,7.0,19.410069,21.237118,22.698757,washtimes,News
3,DS6MrqFEsKc,22.539444,85.0,14.471957,18.505701,21.732696,catloversclub,Non-News
4,DS6QWPnEQfH,31.618889,641.0,17.418103,20.741974,28.533364,msnbc,News


In [20]:
out[out['type'] == 'News']['t95'].describe()

count    43.000000
mean     12.929646
std       5.184807
min       2.895855
25%       9.715249
50%      11.714151
75%      14.975241
max      26.353403
Name: t95, dtype: float64

In [21]:
out[out['type'] == 'Non-News']['t95'].describe()

count     6.000000
mean     13.999367
std       2.737912
min      10.682626
25%      12.326350
50%      13.803061
75%      14.976496
max      18.505701
Name: t95, dtype: float64

In [25]:
out['comments_final'].corr(out['t95'], method='spearman')

np.float64(0.14849969690878212)